# RAISE PDF Parsing + Filtering - Local Upload

Use this notebook in local Jupyter/JupyterLab or VS Code notebooks. Give it a local PDF path, or use the optional upload widget if your notebook frontend supports widgets. It copies the PDF into the local project folder, runs Stage 1 PDF parsing and Stage 2 filtering, then writes the JSON outputs under `data/`.

## Install Local Package

Run this once per environment. It installs the package in editable mode from the module folder. The optional upload widget uses `ipywidgets`, but the plain file-path option works even when widgets are not visible.

In [ ]:
from pathlib import Path
import sys

MODULE_ROOT = Path.cwd()
if MODULE_ROOT.name == "notebooks":
    MODULE_ROOT = MODULE_ROOT.parent
elif (MODULE_ROOT / "RAISE_PDF_Parsing").is_dir():
    MODULE_ROOT = MODULE_ROOT / "RAISE_PDF_Parsing"

PACKAGE_SRC = MODULE_ROOT / "src"
if str(PACKAGE_SRC) not in sys.path:
    sys.path.insert(0, str(PACKAGE_SRC))

print("Module root:", MODULE_ROOT)
print("Package source:", PACKAGE_SRC)

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-e",
    str(MODULE_ROOT),
    "ipywidgets",
])
print("Installed local package and ipywidgets into this kernel environment.")

## Select PDF Locally

Recommended for VS Code/local notebooks: paste the full path to your annual-report PDF below. Leave it empty only if you want to try the optional upload widget in the next cell.

In [ ]:
# Example Linux path: "/home/Ada/Downloads/annual_report.pdf"
# Example Windows path: r"C:\\Users\\Ada\\Downloads\\annual_report.pdf"
# Leave empty if you will use the upload widget below.
LOCAL_PDF_PATH = ""

# Optional per-block JSON Schema. Matching blocks are kept; non-matching blocks are removed.
# Leave empty if you will upload a schema or do not want custom schema filtering.
LOCAL_SCHEMA_PATH = ""

## Optional Upload Widget

If a button appears below, you can upload a PDF instead of using `LOCAL_PDF_PATH`. If nothing appears, skip this cell and use the file-path option above.

In [ ]:
try:
    from IPython.display import display
    import ipywidgets as widgets

    uploader = widgets.FileUpload(
        accept=".pdf,application/pdf",
        multiple=False,
        description="Upload PDF",
    )
    schema_uploader = widgets.FileUpload(
        accept=".json,application/json,application/schema+json",
        multiple=False,
        description="Upload Schema",
    )
    display(uploader, schema_uploader)
except Exception as exc:
    uploader = None
    schema_uploader = None
    print("Upload widget is unavailable. Use LOCAL_PDF_PATH above instead.")
    print(type(exc).__name__ + ":", exc)

## Load Optional Content Schema

The custom schema is evaluated against each Stage 2 block. A block is kept only when it validates. For example, `properties.block_type_guess.enum` can select headings and tables, while `properties.text.pattern` can retain text matching research terms. Invalid schemas stop the run with a clear error.

In [ ]:
import json

def uploaded_bytes(widget):
    if widget is None or not widget.value:
        return None
    value = widget.value
    item = next(iter(value.values())) if isinstance(value, dict) else value[0]
    return bytes(item["content"])

custom_schema = None
if LOCAL_SCHEMA_PATH.strip():
    schema_path = Path(LOCAL_SCHEMA_PATH).expanduser().resolve()
    custom_schema = json.loads(schema_path.read_text(encoding="utf-8"))
    print("Loaded custom schema:", schema_path)
elif schema_uploader is not None and schema_uploader.value:
    custom_schema = json.loads(uploaded_bytes(schema_uploader).decode("utf-8"))
    print("Loaded uploaded custom schema.")
else:
    print("No custom schema selected; only the built-in Stage 2 filters will run.")

if custom_schema is not None:
    from raise_pdf_parsing.schema_filter import validate_custom_schema
    validate_custom_schema(custom_schema)
    print("Custom JSON Schema is valid (Draft 2020-12).")

In [ ]:
import shutil

RAW_DIR = MODULE_ROOT / "data" / "01_raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

if LOCAL_PDF_PATH.strip():
    source_pdf_path = Path(LOCAL_PDF_PATH).expanduser().resolve()
    if not source_pdf_path.exists():
        raise FileNotFoundError(f"PDF path does not exist: {source_pdf_path}")
    if source_pdf_path.suffix.lower() != ".pdf":
        raise ValueError(f"Expected a .pdf file path, got: {source_pdf_path}")

    input_pdf_path = RAW_DIR / source_pdf_path.name
    if source_pdf_path != input_pdf_path.resolve():
        shutil.copy2(source_pdf_path, input_pdf_path)
    print("Copied local PDF to:", input_pdf_path)

elif uploader is not None and uploader.value:
    uploaded_value = uploader.value
    if isinstance(uploaded_value, dict):
        uploaded_file = next(iter(uploaded_value.values()))
        filename = uploaded_file.get("metadata", {}).get("name") or uploaded_file.get("name")
        content = uploaded_file["content"]
    else:
        uploaded_file = uploaded_value[0]
        filename = uploaded_file["name"]
        content = uploaded_file["content"]

    input_pdf_path = RAW_DIR / filename
    input_pdf_path.write_bytes(bytes(content))
    print("Saved uploaded PDF to:", input_pdf_path)

else:
    raise RuntimeError("Set LOCAL_PDF_PATH to a PDF path, or upload a PDF with the widget if it is visible.")

## Run Stage 1 + Stage 2

This creates `report_blocks.json`, `target_blocks.json`, and `filtering_report.json` in the module's `data/` folders.

In [ ]:
import sys

if str(PACKAGE_SRC) not in sys.path:
    sys.path.insert(0, str(PACKAGE_SRC))

from raise_pdf_parsing.main import run_pipeline
from raise_pdf_parsing.models import TargetBlocksDocument
from raise_pdf_parsing.schema_filter import filter_target_by_schema

parsed_output_path = MODULE_ROOT / "data" / "02_parsed" / "report_blocks.json"
target_output_path = MODULE_ROOT / "data" / "03_filtered" / "target_blocks.json"
filtering_report_path = MODULE_ROOT / "data" / "03_filtered" / "filtering_report.json"

parsed, target = run_pipeline(
    pdf_path=input_pdf_path,
    parsed_output_path=parsed_output_path,
    target_output_path=target_output_path,
    filtering_report_path=filtering_report_path,
)

schema_filter_report_path = MODULE_ROOT / "data" / "03_filtered" / "schema_filtering_report.json"
if custom_schema is not None:
    target, schema_filter_report = filter_target_by_schema(target, custom_schema)
    target_payload = target.model_dump(exclude_none=False)
    TargetBlocksDocument.model_validate(target_payload)
    target_output_path.write_text(json.dumps(target_payload, indent=2, ensure_ascii=False), encoding="utf-8")
    schema_filter_report_path.write_text(json.dumps(schema_filter_report, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Custom schema removed {schema_filter_report['removed_block_count']} block(s).")
    print("Validated and rewrote:", target_output_path)

meta = target.filtering_metadata
print(f"Parsed {parsed.page_count} pages, {len(parsed.blocks)} blocks.")
print(f"Kept {meta.output_block_count} blocks across {len(meta.retained_pages)} pages.")
print(f"Excluded {len(meta.excluded_pages)} page(s).")

## Review Filtering Decisions

In [ ]:
for record in target.filtering_metadata.excluded_pages:
    print(f"page {record.page_number}: {record.category} - {record.reason}")

print()
print(f"Removed {target.filtering_metadata.removed_boilerplate_block_count} boilerplate block(s).")

print()
print("First 5 kept blocks:")
for block in target.blocks[:5]:
    print(f"  [{block.page_number}] {block.block_type_guess:<10} {block.text[:80]!r}")

## Output Files

In [ ]:
output_paths = [parsed_output_path, target_output_path, filtering_report_path]
if custom_schema is not None:
    output_paths.append(schema_filter_report_path)
for path in output_paths:
    print(path, "exists=" + str(path.exists()), "size=" + str(path.stat().st_size if path.exists() else 0))